# Attempting to Model NGC6569 with PyfalcON


In [ ]:
import numpy as np

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation

import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 200  # Set to 100MB or whatever you need

from IPython.display import HTML

import pandas as pd

import pyfalcon

In [ ]:
import astropy.coordinates as coord
import astropy.units as u
from astropy.constants import G
import gala.coordinates as gc
import gala.dynamics as gd
import gala.potential as gp
from gala.units import galactic
from gala.dynamics import mockstream as ms

import agama

import importlib
import sys
from pathlib import Path

from time import time

START_DIR = Path.cwd().resolve()
SEARCH_DIRS = (START_DIR, *START_DIR.parents)
NGC6569_DIR = next(
    (path for path in SEARCH_DIRS if (path / "milkyway").is_dir() and (path / "helpers").is_dir()),
    None,
)
if NGC6569_DIR is None:
    NGC6569_DIR = next(
        (path / "joe's_Code" / "ngc6569" for path in SEARCH_DIRS
         if (path / "joe's_Code" / "ngc6569" / "milkyway").is_dir()),
        None,
    )
if NGC6569_DIR is None:
    raise FileNotFoundError("Could not locate joe's_Code/ngc6569 from the current working directory")

DATA_DIR = NGC6569_DIR / "data"
OUTPUT_DIR = NGC6569_DIR / "output"
MILKYWAY_DIR = NGC6569_DIR / "milkyway"
FIGURE_DIR = OUTPUT_DIR / "figures"
ORBIT_FIGURE_DIR = FIGURE_DIR / "orbits"
MASS_LOSS_FIGURE_DIR = FIGURE_DIR / "mass_loss"
OBSERVABLE_FIGURE_DIR = FIGURE_DIR / "observables"
TIDAL_TAIL_FIGURE_DIR = FIGURE_DIR / "tidal_tail"
for _path in (DATA_DIR, OUTPUT_DIR, ORBIT_FIGURE_DIR, MASS_LOSS_FIGURE_DIR, OBSERVABLE_FIGURE_DIR, TIDAL_TAIL_FIGURE_DIR):
    _path.mkdir(parents=True, exist_ok=True)
if str(NGC6569_DIR) not in sys.path:
    sys.path.insert(0, str(NGC6569_DIR))

from helpers.leap_frog import kdk_leapfrog_TD
from helpers.leap_frog import kdk_leapfrog

#from leap_frog_hdf5 import kdk_leapfrog_hdf5, load_simulation_hdf5

In [ ]:
# default Astropy Galactocentric frame parameters to the values adopted in Astropy v4.0:
_ = coord.galactocentric_frame_defaults.set('v4.0')

# set Agama units 
# working units: 1 Msun, 1 kpc, 1 km/s
agama.setUnits(length=1*u.kpc, velocity=1*u.km/u.s, mass=1*u.Msun)
print("Newton G in Agama units,",agama.G)

# Check the current unit system
print("Current Agama units:")
print(f"Length unit: {agama.getUnits()['length']}")
print(f"Velocity unit: {agama.getUnits()['velocity']}")  
print(f"Time unit: {agama.getUnits()['time']}")
print(f"Mass unit: {agama.getUnits()['mass']}")

agama_time_unit = agama.getUnits()["time"]
print(agama_time_unit)

In [ ]:
# Use the Hunter rotating potential 
pot_ext = agama.Potential(str(MILKYWAY_DIR / "MWPotentialHunter24_full.ini")) 
pot_rot = agama.Potential(str(MILKYWAY_DIR / "MWPotentialHunter24_rotating.ini")) 
pot_bovy = agama.Potential(str(MILKYWAY_DIR / "MWPotential2014.ini")) 

pot_use = pot_bovy

In [ ]:
pot_ext

In [ ]:
# NG6569 coordinates 

c = coord.SkyCoord(ra = 273.412*u.degree, dec = -31.827*u.degree,
                        distance=(10.5)*u.kpc,
                        pm_ra_cosdec= -4.125*u.mas/u.yr,
                        pm_dec= -7.354*u.mas/u.yr,
                        radial_velocity= -49.82*u.km/u.s)

# transform to galactic centeric 
c_gc = c.transform_to(coord.Galactocentric).data
print(c_gc._differentials)

In [ ]:
# creat phase space object
w0 = gd.PhaseSpacePosition(c_gc)
print("initial position :", w0.pos)
print("initial velocity :", w0.vel)
print("Need to convert velocity to km/s")

pos_0 = np.r_[w0.pos.x.value, w0.pos.y.value, w0.pos.z.value]
vel_0 = np.r_[w0.vel.d_x.to(u.km/u.s).value, 
              w0.vel.d_y.to(u.km/u.s).value,
              w0.vel.d_z.to(u.km/u.s).value]
print("position", pos_0) 
print("velocity", vel_0) 

In [ ]:
# check against 
# -31.81767578935911 km / s -174.361128405216 km / s 23.931083813686854 km / s

In [ ]:
# # check the enclosed mass at rmax of the NFW profile, so that it seems reasonable 
# rmax = alpha*r_scale
# pot["halo"].mass_enclosed(np.array([rmax.value, 0, 0]))

In [ ]:
# integrate orbit 
tfin= -200*u.Myr
nt=2000
t_eval = np.linspace(0, tfin, nt)
t_scipy = t_eval.to(u.Gyr).value/agama_time_unit.to(u.Gyr).value
t_scipy

In [ ]:
def rhs(t,state): 
    
    pos = state[:3]
    vel = state[3:]

    acc = pot_use.force(pos, t=t)
    return np.r_[vel, acc] 

# integrate with solve_ivp
from scipy.integrate import solve_ivp

state_0 = np.r_[pos_0, vel_0] 
print(state_0, state_0.shape)

t_span = (0, t_scipy[-1]) 
sol = solve_ivp(rhs, t_span, state_0, t_eval =t_scipy, rtol=1e-10, atol=1e-10)
orbit = sol["y"]

In [ ]:
x = orbit[0,:]
y = orbit[1,:]
z = orbit[2,:]
vx = orbit[3,:]
vy = orbit[4,:]
vz = orbit[5,:]

In [ ]:
# Agama orbit
# Calculate orbit
orbit = agama.orbit(potential=pot_use, 
                   ic=state_0, 
                   time=t_span[1],      # Total integration time
                   trajsize=nt)   # Number of output points

In [ ]:
orbit[1].shape

In [ ]:
# 2-D orbit figures 
fig = plt.figure(figsize=(14, 4))

ax = fig.add_subplot(131)
ax.set_xlabel("x [kpc]", size=20) 
ax.set_ylabel("y [kpc]", size=20)
ax.plot(x,y, label="from scratch")
ax.plot(orbit[1][:,0], orbit[1][:,1], "--", label="Agama") 
ax.legend()

ax = fig.add_subplot(132)
ax.set_xlabel("x [kpc]", size=20) 
ax.set_ylabel("z [kpc]", size=20)
ax.plot(x,z)
ax.plot(orbit[1][:,0], orbit[1][:,2], "--") 

ax = fig.add_subplot(133)
ax.set_xlabel("y [kpc]", size=20) 
ax.set_ylabel("z [kpc]", size=20)
ax.plot(y,z)
ax.plot(orbit[1][:,1], orbit[1][:,2], "--") 
plt.tight_layout()
fig.savefig(ORBIT_FIGURE_DIR / "ngc6569_oribit.pdf")

## GyrfalcON manuals
* https://teuben.github.io/nemo/man_html/gyrfalcON.1.html
* 



In [ ]:
# Create figure
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
    
# Create scatter plot with color mapping by radius
ax.plot(x,y,z, alpha = .5) 
ax.set_xlabel('X (pc)')
ax.set_ylabel('Y (pc)')
ax.set_zlabel('Z (pc)')
fig.savefig(ORBIT_FIGURE_DIR / "ngc6569_3d_orbit.pdf") 


In [ ]:
# Define the parameters for your King model
W0_value = 7.0  # Example W0 value
# create an isolated star cluster
r_scale = 1/1000
m = 2.3*1e5*(2)
pot_sat = agama.Potential(type='king', W0=W0_value, scaleRadius=r_scale, mass=m)
df_sat = agama.DistributionFunction(type='quasispherical', potential=pot_sat)
Nbody = 150000
xv, particle_masses = agama.GalaxyModel(pot_sat, df_sat).sample(Nbody)
mass = particle_masses  # Backwards-compatible alias for older cells.

r_agama = np.sqrt(xv[:,0]**2 + xv[:,1]**2 + xv[:,2]**2) 
v_agama = np.sqrt(xv[:,3]**2 + xv[:,4]**2 + xv[:,5]**2) 

print("Agama G:", agama.G)

cluster_data = np.c_[particle_masses, xv]
cluster_data.shape
np.savetxt(DATA_DIR / "cluster_data.txt", cluster_data)

In [ ]:

# look at positions in 3D
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
    
# Create scatter plot
ax.scatter(xv[:,0], xv[:,1], xv[:,2], alpha=0.1, s=20)
    
# Labels and title
ax.set_xlabel('X [kpc]', fontsize=12)
ax.set_ylabel('Y [kpc]', fontsize=12)
ax.set_zlabel('Z [kpc]', fontsize=12)
    

In [ ]:
fig = plt.figure(figsize=(14,6))

ax=fig.add_subplot(121)
ax.set_xlabel("radial distance", size=20) 
ax.hist(r_agama, bins=30, edgecolor="black", alpha=.5, label ="Agama")

ax.axvline(r_scale, color="black", label="core radius")
ax.legend(loc=0) 

ax=fig.add_subplot(122)
ax.set_xlabel("speed", size=20) 
ax.hist(v_agama, bins=30, edgecolor="black", alpha=.5, label ="Agama")

ax.legend(loc=0) 


In [ ]:
pos_0 = np.r_[x[-1], y[-1], z[-1]]
vel_0 = np.r_[vx[-1], vy[-1], vz[-1]]
print(pos_0)
print(vel_0) 

In [ ]:
print("make this a function")

def shift_to_gc(xv, pos_0, vel_0): 

    x = xv[:,0] + pos_0[0]
    y = xv[:,1] + pos_0[1]
    z = xv[:,2] + pos_0[2]

    vx = xv[:,3] + vel_0[0]
    vy = xv[:,4] + vel_0[1]
    vz = xv[:,5] + vel_0[2]


    print("checks")
    print(np.mean(x), np.mean(y), np.mean(z))
    print(pos_0)


    print(np.mean(vx), np.mean(vy), np.mean(vz))
    print(vel_0)

    return np.column_stack((x, y, z)), np.column_stack((vx, vy, vz))


In [ ]:
# pos_0 = np.column_stack((x, y, z))
# vel_0 = np.column_stack((vx, vy, vz))
# vel_0.shape

pos_0 , vel_0 = shift_to_gc(xv, pos_0, vel_0)

In [ ]:
np.sum(mass/1e5)
np.savetxt(DATA_DIR / "ngc6569_mass.txt", mass)

In [ ]:
time_unit=u.kpc.to(u.km)*u.s.to(u.Gyr)
time_unit

In [ ]:
tmax = -tfin.to(u.Gyr).value/time_unit
print("maximum time", tmax)
print(t_scipy[-1])

In [ ]:
kmax=16
tau = 2**(-kmax)*time_unit
print("time step:", tau) 
nt=int(tmax/tau) + 1
print("number of time steps:", nt)

eps = 1/1000  # II 
eps = .1/1000 # I 
eps_power = -4
eps = (2**eps_power)/1000
print("softening length:", eps) 

In [ ]:
(nt)*tau

In [ ]:
print(nt)

In [ ]:
downsample=20
#filename="ngc_6569_runI"
#nt=1000

In [ ]:
pot_use

In [ ]:
t1 = time()
sim_data = kdk_leapfrog(pot_use, pos_0, vel_0, 
                        mass, nt, tau, agama.G, eps, 
                        time_unit, downsample,last_snapshot=False)
t2=time()
print("run time", t2-t1, (t2-t1)/60)

In [ ]:
len(sim_data)

In [ ]:
#gc_data = load_simulation_hdf5(sim_data)

In [ ]:
#gc_data["metadata"]

In [ ]:
# get time array 
time_array=[]
for i in range(len(sim_data)): 

    t = sim_data[i]["time"]
    time_array.append(t*1000) 

time_array=np.array(time_array)    

In [ ]:
from helpers.bound_funcs import get_bound_particles

bound_data = get_bound_particles(sim_data, mass)

In [ ]:
# Create figure
fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111)  # Fixed: added subplot number
ax.set_aspect('equal')     # Fixed: proper way to set equal aspect
ax.set_xlabel('X (kpc)')
ax.set_ylabel('Y (kpc)')
L = 4
ax.set_xlim(-L, L)
ax.set_ylim(-L, L)

ax.plot(orbit[1][:,0],orbit[1][:,1], color="black", alpha=.1, label="Agama")
# gala orbit 
#ax.plot(orbit.pos.x,orbit.pos.y, color="black", alpha=.5, label="Gala")
pts, = ax.plot([], [], "o", color="blue", alpha=0.25, markersize=1)  # Added markersize for visibility
plt.close()

# Initialization function
def init():
    pts.set_data([], []) 
    return pts, 

def draw(i):  # Fixed: use 'i' instead of undefined 'nt'

    data = sim_data[i]
    pos = data["pos"]


    x = pos[:,0]
    y = pos[:,1]
    z = pos[:,2] 
    
    pts.set_data(x, y)  # Fixed: missing closing bracket and proper syntax

    time = np.round(time_array[i] + tfin.value,0) 
    
    ax.set_title(f'NGC6569 5 (X-Y Plane) - Time: {time} Myr')  # Fixed: use 'i' instead of 'nt'
    return pts, 
    
anim = FuncAnimation(fig, draw, init_func=init, frames=len(sim_data), interval=50, blit=True)
anim.save(str(OUTPUT_DIR / "6569_2d_II.mp4")) 
HTML(anim.to_jshtml())

In [ ]:
len(sim_data)

In [ ]:
def trajectories(bound_data, sim_data): 
    """
    Extract time evolution trajectories of bound cluster properties from simulation data.
    
    This function processes the output from bound particle tracking to create clean
    time series arrays for analysis and plotting of cluster evolution.
    
    Parameters:
    -----------
    bound_data : list of dict
        List of dictionaries containing bound particle data at each timestep.
        Each dictionary should contain keys: 'total mass', 'total energy', 'pos', 'vel'
    sim_data : list of dict
        List of dictionaries containing simulation data at each timestep.
        Each dictionary should contain key: 'time'
        
    Returns:
    --------
    dict : Dictionary containing trajectory arrays
        'mass' : array, shape (n,) - Total mass of bound particles vs time
        'energy' : array, shape (n,) - Total energy of bound particles vs time  
        'time' : array, shape (n,) - Time array
        'pos' : array, shape (n, 3) - Center of mass position vs time
        'vel' : array, shape (n, 3) - Center of mass velocity vs time

    """
    
    n = len(bound_data)
    
    # Validate input lengths match
    if len(sim_data) != n:
        raise ValueError(f"Length mismatch: bound_data has {n} entries, sim_data has {len(sim_data)}")
    
    # Initialize trajectory arrays
    time_array = np.zeros(n)
    mass_traj = np.zeros(n)
    energy_traj = np.zeros(n)
    r_traj = np.zeros((n, 3))
    d_traj = np.zeros(n)
    v_traj = np.zeros((n, 3))
    
    # Extract data for each timestep
    for i in range(n): 
        # Extract bound particle properties
        mass_traj[i] = bound_data[i]["total mass"]
        energy_traj[i] = bound_data[i]["total energy"]
        r_traj[i, :] = bound_data[i]["pos"]
        v_traj[i, :] = bound_data[i]["vel"]
        d_traj[i] = np.linalg.norm(r_traj[i,:]) 
        
        # Extract time from simulation data
        time_array[i] = sim_data[i]["time"] 
    
    # Package results into dictionary
    traj = {
        'mass': mass_traj,
        'energy': energy_traj,
        'time': time_array,
        'pos': r_traj,
        'vel': v_traj,
        'distance': d_traj
    }
    
    return traj

traj = trajectories(bound_data, sim_data)

In [ ]:
# mass_traj = []
# traj =[] 
# v_traj =[]

# for i in range(len(bound_data)): 

#     m = bound_data[i]["total mass"]
#     pos = bound_data[i]["pos"]
#     vel = bound_data[i]["vel"]
#     mass_traj.append(m) 
#     traj.append(pos) 
#     v_traj.append(vel) 

# mass_traj=np.array(mass_traj)/np.sum(mass)
# traj = np.array(traj)
# v_traj = np.array(v_traj)


# fig = plt.figure()
# fig.suptitle('Mass Loss', size=30)
# ax = fig.add_subplot()
# ax.set_ylabel(r"$M/M_0$", size=20) 
# ax.set_xlabel("Myr", size=20) 
# ax.plot(time_array + tfin.value, mass_traj)

In [ ]:
def cluster_frame(r_cm, v_cm, pos, vel): 
    """
    Compute unit vectors for cluster frame and transform positions to that frame.
    
    Parameters:
    -----------
    r_cm : array, shape (3,)
        Position vector from galactic center to cluster center of mass
    v_cm : array, shape (3,)
        Velocity vector of cluster center of mass  
    pos : array, shape (N, 3)
        Positions of N particles
    vel : array, shape (N, 3)
        Velocities of N particles
        
    Returns:
    --------
    tuple : (x_hat, y_hat, z_hat, pos_cluster_frame, vel_cluster_frame, omega_vec)
        x_hat, y_hat, z_hat : unit vectors in cluster frame
        pos_cluster_frame : array, shape (N, 3) - positions in cluster frame
        vel_cluster_frame : array, shape (N, 3) - velocities in cluster frame
        omega_vec : array, shape (3,) - angular velocity vector of cluster
    """
    
    # x-unit vector points toward galactic center 
    x_hat = -r_cm / np.linalg.norm(r_cm)
    
    # z-unit vector is perpendicular to orbital plane (r × v direction)
    L_vec = np.cross(r_cm, v_cm)
    z_hat = L_vec / np.linalg.norm(L_vec)
    
    # y-unit vector completes right-handed system (z × x)
    y_hat = np.cross(z_hat, x_hat)
    
    # Angular velocity vector: ω = L / |r|²
    r_cm_mag_sq = np.dot(r_cm, r_cm)
    omega_vec = L_vec / r_cm_mag_sq
    
    # Transform positions to cluster frame
    # Each row of pos gets dotted with each unit vector
    x_coords = np.dot(pos, x_hat)  # shape (N,)
    y_coords = np.dot(pos, y_hat)  # shape (N,)
    z_coords = np.dot(pos, z_hat)  # shape (N,)
    
    # Stack coordinates to form (N, 3) array
    pos_cluster_frame = np.column_stack([x_coords, y_coords, z_coords])
    
    # Transform velocities to cluster frame
    # Each row of vel gets dotted with each unit vector
    vx_coords = np.dot(vel, x_hat)  # shape (N,)
    vy_coords = np.dot(vel, y_hat)  # shape (N,)
    vz_coords = np.dot(vel, z_hat)  # shape (N,)
    
    # Stack velocity components to form (N, 3) array
    vel_cluster_frame = np.column_stack([vx_coords, vy_coords, vz_coords])
    
    return x_hat, y_hat, z_hat, pos_cluster_frame, vel_cluster_frame, omega_vec
        
  

In [ ]:
def get_cluster_data(bound_data, sim_data): 
    n=len(bound_data) 

    cluster_data = []

    # angular velocity of cluster
    omega_c = np.zeros((n, 3))

    # unit vectors in cluster frame 
    for i in range(n): 

        b_data = bound_data[i]
        r_cm = b_data["pos"]
        v_cm = b_data["vel"]

        pos = sim_data[i]["pos"]
        vel = sim_data[i]["vel"]

        obj = cluster_frame(r_cm, v_cm, pos, vel)

        x_hat, y_hat, z_hat, pos_cluster_frame, vel_cluster_frame, omega_vec = obj
        
        c_data={}
        
        c_data["omega"] = omega_vec
        c_data["x_hat"] = x_hat
        c_data["y_hat"] = y_hat
        c_data["z_hat"] = z_hat 
        c_data["pos"]= pos_cluster_frame
        c_data["vel"]=vel_cluster_frame

        cluster_data.append(c_data)
 
    return cluster_data

cluster_data= get_cluster_data(bound_data, sim_data) 

In [ ]:
def get_tidal_data(traj, pot_ext): 

    pos = traj["pos"]
    n=pos.shape[0]

    tidal = np.zeros((n,3))
    max_tidal = np.zeros(n)

    for i in range(n): 

        H = np.zeros((3, 3))
        
        hess = pot_ext.eval(pos[i,:], der=True)
    
    
        # Upper triangular indices
        triu_indices = np.triu_indices(3)
        H[triu_indices] = hess
    
        # Make symmetric
        H = H + H.T - np.diag(np.diag(H))

        tidal[i,:] = np.linalg.eigvals(H)

        max_tidal[i] = np.max(tidal[i,:]) 

    return tidal, max_tidal 

tidal, max_tidal = get_tidal_data(traj, pot_ext)

In [ ]:
# # tidal tensor analysis

# n = traj["pos"].shape[0]
# tidal = np.zeros((n, 3))
# max_tidal=np.zeros(n)
# for i in range(n): 

#     H = np.zeros((3, 3))

#     point = traj[i]["pos"]
    
#     hess = pot_ext.eval(point, der=True)
#     #print(eigen_vals)
#     #print(hess)
    
#     # Upper triangular indices
#     triu_indices = np.triu_indices(3)
#     H[triu_indices] = hess
    
#     # Make symmetric
#     H = H + H.T - np.diag(np.diag(H))

#     tidal[i,:] = np.linalg.eigvals(H)

#     max_tidal[i] = np.max(tidal[i,:]) 
    

In [ ]:
gc_energy = np.zeros(len(bound_data))

i=0
for bdata in bound_data: 

    e = bdata["total energy"]
    gc_energy[i] = e
    i+=1

In [ ]:
#traj["time"]

In [ ]:

max_t=np.max(max_tidal)

fig = plt.figure(figsize=(14, 8 ))
fig.suptitle('Mass Loss, Energy, Max Tidal Force, Radius ', size=20)
ax = fig.add_subplot(411)
ax.set_xlim(tfin.value, 0)
#ax.set_xlim(tfin.value, -900)
ax.set_ylabel(r"$M/M_0$", size=20) 
#ax.set_xlabel("Myr", size=20)

M0=traj["mass"][0]
ax.plot(time_array+tfin.value, traj["mass"]/M0)


ax = fig.add_subplot(412)
ax.set_xlim(tfin.value, 0)
#ax.set_xlim(tfin.value, -900)
ax.set_ylabel(r"$E/|E_0|$", size=20) 
#ax.set_xlabel("Myr", size=20) 
E = traj["energy"]
ax.plot(time_array+tfin.value, E/np.abs(E[0])) 


ax = fig.add_subplot(413)
ax.set_xlim(tfin.value, 0)
#ax.set_xlim(tfin.value, -900)
ax.set_ylabel(r"$\lambda_{max}$ ", size=20) 
ax.set_xlabel("Myr", size=20) 
ax.plot(time_array + tfin.value, max_tidal)


ax = fig.add_subplot(414)
ax.set_xlim(tfin.value, 0)
#ax.set_xlim(tfin.value, -900)
ax.set_ylabel(r"$r(t)$ [kpc]", size=20) 
ax.set_xlabel("Myr", size=20) 
ax.plot(time_array + tfin.value, traj["distance"])

fig.savefig(MASS_LOSS_FIGURE_DIR / "6569_ml_II.pdf") 

# print("Maybe add distance from galactic center." ) 
# print("How does tidal stripping compare at large tidal tensor versus Particle Spray method.") 
# print("why is there a dip in energy right before an increase?" ) 


In [ ]:
# Mass loss by itself
fig, axes = plt.subplots(1, 1, figsize=(8, 5))
axes.set_xlim(tfin.value, 0)
axes.set_xlabel("Myr", size=14)
axes.set_ylabel(r"$M/M_0$", size=14)
axes.set_title('Cluster Bound Mass Loss', size=16)
axes.plot(time_array + tfin.value, traj["mass"] / M0)
fig.tight_layout()
fig.savefig(MASS_LOSS_FIGURE_DIR / "6569_mass_loss.pdf")

In [ ]:
r_traj = np.linalg.norm(traj["pos"], axis=1)
r_traj.shape

In [ ]:
# Create figure
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')

x = traj["pos"][:,0]
y = traj["pos"][:,1]
z = traj["pos"][:,2]
    
# Create scatter plot with color mapping by radius
ax.plot(x,y,z, alpha = .5) 
ax.set_xlabel('X (pc)')
ax.set_ylabel('Y (pc)')
ax.set_zlabel('Z (pc)')


## PeTar Matching / Notebook-Run Comparisons

This section compares only runs produced by this notebook session:

- the main notebook run (`sim_data`, `bound_data`, `traj`), and
- any additional `kmax` / `eps_pc` experiments you run below.

No precomputed 1 Gyr reference bundle is used here. The first available run is treated as the comparison baseline unless you set `BASELINE_LABEL` to another run label.

In [ ]:
# Experiment paths and helpers for notebook-run comparisons only.
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "NGC6569_DIR" not in globals():
    START_DIR = Path.cwd().resolve()
    SEARCH_DIRS = (START_DIR, *START_DIR.parents)
    NGC6569_DIR = next(
        (path for path in SEARCH_DIRS if (path / "milkyway").is_dir() and (path / "helpers").is_dir()),
        None,
    )
    if NGC6569_DIR is None:
        NGC6569_DIR = next(
            (path / "joe's_Code" / "ngc6569" for path in SEARCH_DIRS
             if (path / "joe's_Code" / "ngc6569" / "milkyway").is_dir()),
            None,
        )
    if NGC6569_DIR is None:
        raise FileNotFoundError("Could not locate joe's_Code/ngc6569 from the current working directory")

    DATA_DIR = NGC6569_DIR / "data"
    OUTPUT_DIR = NGC6569_DIR / "output"
    MILKYWAY_DIR = NGC6569_DIR / "milkyway"

PETAR_MATCH_OUT = OUTPUT_DIR / "petar_matching_experiments"
PETAR_MATCH_FIGURE_DIR = PETAR_MATCH_OUT / "figures"
PETAR_MATCH_CACHE = PETAR_MATCH_OUT / "cache"
for _path in (PETAR_MATCH_OUT, PETAR_MATCH_FIGURE_DIR, PETAR_MATCH_CACHE):
    _path.mkdir(parents=True, exist_ok=True)

PARTICLE_COLUMNS = ["mass", "x_gc", "y_gc", "z_gc", "vx_gc", "vy_gc", "vz_gc"]
POSITION_COLUMNS = ["x_gc", "y_gc", "z_gc"]
VELOCITY_COLUMNS = ["vx_gc", "vy_gc", "vz_gc"]


def _manifest_path(path):
    path = Path(path)
    return path.with_suffix(".json")


def read_binary_table(path):
    path = Path(path)
    meta_path = _manifest_path(path)
    if not meta_path.exists():
        raise FileNotFoundError(f"Missing sidecar manifest: {meta_path}")
    meta = json.loads(meta_path.read_text())
    arr = np.fromfile(path, dtype=np.dtype(meta["dtype"])).reshape(meta["shape"])
    return arr, meta


def write_binary_table(path, data, columns, units=None, description="", extra_meta=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    arr = np.asarray(data, dtype=np.float64)
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    arr.tofile(path)
    meta = {
        "binary_file": path.name,
        "dtype": "float64",
        "shape": list(arr.shape),
        "columns": list(columns),
        "units": units or {},
        "description": description,
    }
    if extra_meta:
        meta.update(extra_meta)
    _manifest_path(path).write_text(json.dumps(meta, indent=2))
    return path


def frame_from_table(data, meta):
    return pd.DataFrame(data, columns=meta["columns"])


def particles_from_snapshot(snapshot, masses):
    pos = np.asarray(snapshot["pos"])
    vel = np.asarray(snapshot["vel"])
    masses = np.asarray(masses, dtype=float).reshape(-1)
    if pos.shape != vel.shape or pos.ndim != 2 or pos.shape[1] != 3:
        raise ValueError("snapshot pos/vel must both have shape (N, 3)")
    if masses.size != pos.shape[0]:
        raise ValueError(f"masses length {masses.size} does not match particle count {pos.shape[0]}")
    return pd.DataFrame(np.column_stack((masses, pos, vel)), columns=PARTICLE_COLUMNS)


def _particle_masses_from_globals():
    if "pos_0" not in globals():
        raise RuntimeError("Run the setup/initial-condition cells first. Missing: pos_0")
    n_particles = np.asarray(pos_0).shape[0]
    candidates = []
    if "particle_masses" in globals():
        candidates.append(("particle_masses", particle_masses))
    if "mass" in globals():
        candidates.append(("mass", mass))
    if "cluster_data" in globals():
        cluster_arr = np.asarray(cluster_data)
        if cluster_arr.ndim == 2 and cluster_arr.shape[1] >= 1:
            candidates.append(("cluster_data[:, 0]", cluster_arr[:, 0]))

    for source, values in candidates:
        arr = np.asarray(values, dtype=float).reshape(-1)
        if arr.size == 1:
            return np.full(n_particles, arr[0], dtype=float)
        if arr.size == n_particles:
            return arr

    details = ", ".join(f"{source}: shape={np.shape(values)}" for source, values in candidates) or "none"
    raise ValueError(f"Could not find per-particle masses matching pos_0 length {n_particles}. Candidates: {details}")


def simple_trajectories(bound_data, sim_data):
    n = len(bound_data)
    return {
        "time": np.array([sim_data[i]["time"] for i in range(n)]),
        "mass": np.array([bound_data[i]["total mass"] for i in range(n)]),
        "energy": np.array([bound_data[i]["total energy"] for i in range(n)]),
        "pos": np.array([bound_data[i]["pos"] for i in range(n)]),
        "vel": np.array([bound_data[i]["vel"] for i in range(n)]),
    }


def mass_loss_curve_from_traj(traj, label):
    time_myr = np.asarray(traj["time"]) * 1000.0
    mass_arr = np.asarray(traj["mass"])
    return pd.DataFrame({
        "elapsed_time_Myr": time_myr,
        "bound_mass_fraction": mass_arr / mass_arr[0],
        "bound_mass": mass_arr,
        "label": label,
    })


def current_notebook_result(label="main notebook run"):
    if "sim_data" not in globals() or len(sim_data) == 0:
        return None
    masses = _particle_masses_from_globals()

    result = {
        "label": label,
        "final_particles": particles_from_snapshot(sim_data[-1], masses),
        "payload": {
            "source": "main notebook variables",
            "kmax": globals().get("kmax"),
            "eps_kpc": globals().get("eps"),
            "downsample": globals().get("downsample"),
            "Nbody": int(len(masses)),
            "cluster_mass_Msun": float(np.sum(masses)),
        },
    }

    local_traj = globals().get("traj")
    if local_traj is None and "bound_data" in globals():
        local_traj = simple_trajectories(bound_data, sim_data)
    if local_traj is not None:
        result["traj"] = local_traj
        result["mass_loss"] = mass_loss_curve_from_traj(local_traj, label)
    return result


if "experiment_results" not in globals():
    experiment_results = []

print("Notebook-run comparison helpers loaded.")
print("Run the main simulation cells above first, then run this section to compare it with additional experiments.")

### Build the Comparison Set

Run this after the main simulation has produced `sim_data`. Optional experiments run later will append to `experiment_results` and appear in the same plots.

In [ ]:
def collect_comparison_results(include_main=True):
    results = []
    if include_main:
        main = current_notebook_result()
        if main is not None:
            results.append(main)
    results.extend(experiment_results)
    return results


def result_map(results):
    return {result["label"]: result for result in results}


comparison_results = collect_comparison_results(include_main=True)
if comparison_results:
    print("Comparison runs:")
    for result in comparison_results:
        ml = "mass_loss" in result
        fp = result["final_particles"].shape if "final_particles" in result else None
        print(f"- {result['label']}: final_particles={fp}, mass_loss={ml}, payload={result.get('payload', {})}")
else:
    print("No runs found yet. Run the main simulation cells above, or run experiments below.")

### Position / Velocity Histograms

Overlays the particle distributions for whatever runs are currently in `comparison_results`.

In [ ]:
def sample_particles(df, n=50000, seed=42):
    if len(df) <= n:
        return df
    return df.sample(n=n, random_state=seed)


def plot_component_histograms(results, columns=None, bins=90, sample_n=80000, save_name=None):
    if not results:
        print("No comparison runs available.")
        return None
    columns = columns or (POSITION_COLUMNS + VELOCITY_COLUMNS)
    fig, axes = plt.subplots(2, 3, figsize=(15, 7))
    axes = axes.ravel()
    for ax, col in zip(axes, columns):
        for result in results:
            sample = sample_particles(result["final_particles"], n=sample_n)
            ax.hist(sample[col], bins=bins, density=True, histtype="step", linewidth=1.5, label=result["label"])
        ax.set_xlabel(col)
        ax.set_ylabel("density")
    axes[0].legend(fontsize=9)
    fig.suptitle("Notebook Run Particle Position and Velocity Histograms")
    fig.tight_layout()
    if save_name:
        fig.savefig(PETAR_MATCH_FIGURE_DIR / save_name, bbox_inches="tight")
    return fig

plot_component_histograms(comparison_results, save_name="notebook_run_pos_vel_histograms.pdf");

### Final-Time Phase-Space Planes

This is the morphology check. If the strings/tails in `x-y`, `x-z`, and `y-z` do not look similar between runs, record that parameter set before continuing.

In [ ]:
def plot_phase_space_planes(results, sample_n=50000, seed=123, alpha=0.12, save_name=None):
    if not results:
        print("No comparison runs available.")
        return None
    pairs = [
        ("x_gc", "y_gc"), ("x_gc", "z_gc"), ("y_gc", "z_gc"),
        ("vx_gc", "vy_gc"), ("vx_gc", "vz_gc"), ("vy_gc", "vz_gc"),
    ]
    colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple", "tab:brown", "tab:pink", "tab:gray"]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.ravel()
    for ax, (xcol, ycol) in zip(axes, pairs):
        for result, color in zip(results, colors):
            sample = sample_particles(result["final_particles"], n=sample_n, seed=seed)
            ax.scatter(sample[xcol], sample[ycol], s=1, alpha=alpha, label=result["label"], color=color, rasterized=True)
        ax.set_xlabel(xcol)
        ax.set_ylabel(ycol)
    axes[0].legend(markerscale=6, fontsize=9)
    fig.suptitle("Notebook Run Final Phase-Space Distributions")
    fig.tight_layout()
    if save_name:
        fig.savefig(PETAR_MATCH_FIGURE_DIR / save_name, bbox_inches="tight")
    return fig

plot_phase_space_planes(comparison_results, save_name="notebook_run_final_phase_space_planes.pdf");

### Mass-Loss Histories

The first available run is the default baseline for quantitative mass-loss differences. Change `BASELINE_LABEL` if you want another run to define the comparison target.

In [ ]:
BASELINE_LABEL = None  # e.g. "main notebook run" or "k16_eps0p0625"


def plot_mass_loss_curves(results, title="Notebook Run Bound Mass-Loss Histories", save_name=None):
    curves = [result for result in results if "mass_loss" in result]
    if not curves:
        print("No mass-loss curves available. Run bound_data/traj for at least one run.")
        return None
    fig, ax = plt.subplots(figsize=(8, 5))
    for result in curves:
        curve = result["mass_loss"]
        ax.plot(curve["elapsed_time_Myr"], curve["bound_mass_fraction"], label=result["label"])
    ax.set_xlabel("elapsed time [Myr]")
    ax.set_ylabel(r"$M/M_0$")
    ymin = min(result["mass_loss"]["bound_mass_fraction"].min() for result in curves)
    ymax = max(result["mass_loss"]["bound_mass_fraction"].max() for result in curves)
    pad = max((ymax - ymin) * 0.1, 1e-4)
    ax.set_ylim(ymin - pad, ymax + pad)
    ax.set_title(title)
    ax.legend(fontsize=9)
    fig.tight_layout()
    if save_name:
        fig.savefig(PETAR_MATCH_FIGURE_DIR / save_name, bbox_inches="tight")
    return fig


def score_curve_against_baseline(curve, baseline_curve):
    t_base = baseline_curve["elapsed_time_Myr"].to_numpy()
    y_base = baseline_curve["bound_mass_fraction"].to_numpy()
    t = curve["elapsed_time_Myr"].to_numpy()
    y = curve["bound_mass_fraction"].to_numpy()
    t_min = max(np.nanmin(t), np.nanmin(t_base))
    t_max = min(np.nanmax(t), np.nanmax(t_base))
    use = (t_base >= t_min) & (t_base <= t_max)
    if np.count_nonzero(use) < 2:
        return {"rms_mass_fraction_diff": np.nan, "max_abs_diff": np.nan, "final_abs_diff": np.nan}
    y_interp = np.interp(t_base[use], t, y)
    diff = y_interp - y_base[use]
    return {
        "rms_mass_fraction_diff": float(np.sqrt(np.mean(diff**2))),
        "max_abs_diff": float(np.max(np.abs(diff))),
        "final_abs_diff": float(abs(y_interp[-1] - y_base[use][-1])),
    }


def mass_loss_score_table(results, baseline_label=None):
    curves = [result for result in results if "mass_loss" in result]
    if len(curves) < 2:
        return pd.DataFrame()
    labels = [result["label"] for result in curves]
    baseline_label = baseline_label or labels[0]
    baseline = next(result for result in curves if result["label"] == baseline_label)
    rows = []
    for result in curves:
        scores = score_curve_against_baseline(result["mass_loss"], baseline["mass_loss"])
        rows.append({
            "label": result["label"],
            "baseline": baseline_label,
            **result.get("payload", {}),
            **scores,
        })
    return pd.DataFrame(rows)

plot_mass_loss_curves(comparison_results, save_name="notebook_run_mass_loss_comparison.pdf")
score_table = mass_loss_score_table(comparison_results, BASELINE_LABEL)
if len(score_table):
    score_table.to_csv(PETAR_MATCH_OUT / "notebook_run_mass_loss_scores.csv", index=False)
    display(score_table)
else:
    print("Need at least two runs with mass-loss curves to compute score differences.")

### Optional: Run Additional Experiments

Leave `RUN_MISSING_EXPERIMENTS = False` until you are ready for expensive runs. These experiments use the current initial particle realization (`pos_0`, `vel_0`, `mass`) from the setup cells above, so timestep/softening changes are isolated from random IC changes.

Suggested first pass:

- timestep: `kmax = 13, 14, 15, 16` at fixed `eps_pc`,
- softening: `eps_pc = 0.0625, 0.25, 0.5, 1, 2, 4` at fixed `kmax`.

After each run, rerun the comparison-set, histogram, phase-space, and mass-loss cells above.

In [149]:
import gc
EXPERIMENTS_TO_RUN = [
    # --- baseline (matches the main notebook run) ---
    {"kmax": 16, "eps_pc": 0.0625, "label": "k16_eps0p0625"},

    # --- timestep sweep: eps fixed at baseline, vary kmax (tau = 2**-kmax) ---
    {"kmax": 10, "eps_pc": 0.0625, "label": "k10_eps0p0625"},
    {"kmax": 11, "eps_pc": 0.0625, "label": "k11_eps0p0625"},
    {"kmax": 12, "eps_pc": 0.0625, "label": "k12_eps0p0625"},
    {"kmax": 13, "eps_pc": 0.0625, "label": "k13_eps0p0625"},
    {"kmax": 14, "eps_pc": 0.0625, "label": "k14_eps0p0625"},
    {"kmax": 15, "eps_pc": 0.0625, "label": "k15_eps0p0625"},
    {"kmax": 17, "eps_pc": 0.0625, "label": "k17_eps0p0625"},
    {"kmax": 18, "eps_pc": 0.0625, "label": "k18_eps0p0625"},

    # --- softening sweep: kmax fixed at baseline, vary eps_pc ---
    {"kmax": 16, "eps_pc": 0.0078125, "label": "k16_eps0p0078125"},
    {"kmax": 16, "eps_pc": 0.015625, "label": "k16_eps0p015625"},
    {"kmax": 16, "eps_pc": 0.03125, "label": "k16_eps0p03125"},
    {"kmax": 16, "eps_pc": 0.125, "label": "k16_eps0p125"},
    {"kmax": 16, "eps_pc": 0.25, "label": "k16_eps0p25"},
    {"kmax": 16, "eps_pc": 0.5, "label": "k16_eps0p5"},
    {"kmax": 16, "eps_pc": 1.0, "label": "k16_eps1p0"},
    {"kmax": 16, "eps_pc": 2.0, "label": "k16_eps2p0"},
    {"kmax": 16, "eps_pc": 4.0, "label": "k16_eps4p0"},
    {"kmax": 16, "eps_pc": 8.0, "label": "k16_eps8p0"},

    # --- cross combos: check whether timestep/softening effects interact ---
    {"kmax": 13, "eps_pc": 0.25, "label": "k13_eps0p25"},
    {"kmax": 13, "eps_pc": 1.0, "label": "k13_eps1p0"},
    {"kmax": 11, "eps_pc": 0.25, "label": "k11_eps0p25"},
    {"kmax": 18, "eps_pc": 0.25, "label": "k18_eps0p25"},
]


def experiment_cache_key(payload):
    return hashlib.md5(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:12]


def require_names(*names):
    missing = [name for name in names if name not in globals()]
    if missing:
        raise RuntimeError("Run the setup/initial-condition cells first. Missing: " + ", ".join(missing))


def run_current_ic_experiment(kmax, eps_pc, label=None, use_cache=True):
    require_names("pos_0", "vel_0", "pot_use", "time_unit", "tmax", "downsample", "agama", "kdk_leapfrog")
    from helpers.bound_funcs import get_bound_particles
    masses = _particle_masses_from_globals()

    payload = {
        "kmax": int(kmax),
        "eps_pc": float(eps_pc),
        "Nbody": int(len(masses)),
        "cluster_mass_Msun": float(np.sum(masses)),
        "tmax_native": float(tmax),
        "downsample": int(downsample),
        "source": "current notebook initial conditions",
    }
    key = experiment_cache_key(payload)
    label = label or f"k{kmax}_eps{eps_pc:g}pc_{key}"
    mass_loss_path = PETAR_MATCH_CACHE / f"{label}_mass_loss.bin"
    final_path = PETAR_MATCH_CACHE / f"{label}_final_particles.bin"

    if use_cache and mass_loss_path.exists() and final_path.exists():
        ml, ml_meta = read_binary_table(mass_loss_path)
        final, final_meta = read_binary_table(final_path)
        return {
            "label": label,
            "payload": payload,
            "mass_loss": frame_from_table(ml, ml_meta),
            "final_particles": frame_from_table(final, final_meta),
            "cache_hit": True,
        }

    tau_run = 2 ** (-int(kmax)) * time_unit
    nt_run = int(tmax / tau_run) + 1
    eps_kpc = float(eps_pc) / 1000.0
    print(f"running {label}: kmax={kmax}, tau={tau_run:g}, nt={nt_run}, eps={eps_kpc:g} kpc")

    t_start = time() if "time" in globals() else None
    sim_run = kdk_leapfrog(
        pot_use, pos_0, vel_0, masses, nt_run, tau_run,
        agama.G, eps_kpc, time_unit, downsample, last_snapshot=True,
    )
    if t_start is not None:
        print(f"run time: {(time() - t_start) / 60:.2f} minutes")

    bound_run = get_bound_particles(sim_run, masses)
    traj_func = trajectories if "trajectories" in globals() else simple_trajectories
    traj_run = traj_func(bound_run, sim_run)

    result = {
        "label": label,
        "payload": payload,
        "sim_data": sim_run,
        "bound_data": bound_run,
        "traj": traj_run,
        "mass_loss": mass_loss_curve_from_traj(traj_run, label),
        "final_particles": particles_from_snapshot(sim_run[-1], masses),
        "cache_hit": False,
    }

    write_binary_table(
        mass_loss_path,
        result["mass_loss"][["elapsed_time_Myr", "bound_mass_fraction", "bound_mass"]].to_numpy(),
        ["elapsed_time_Myr", "bound_mass_fraction", "bound_mass"],
        units={"elapsed_time_Myr": "Myr", "bound_mass": "Msun"},
        description="Mass-loss history from notebook-run matching experiment.",
        extra_meta={"cache_payload": payload},
    )
    write_binary_table(
        final_path,
        result["final_particles"].to_numpy(),
        result["final_particles"].columns,
        units={"mass": "Msun", "x_gc": "kpc", "y_gc": "kpc", "z_gc": "kpc", "vx_gc": "km/s", "vy_gc": "km/s", "vz_gc": "km/s"},
        description="Final particle snapshot from notebook-run matching experiment.",
        extra_meta={"cache_payload": payload},
    )

    # Drop the full per-timestep snapshots once they're saved to disk -- keeping
    # sim_data/bound_data/traj (all Nbody x n_snapshots) in memory for every run
    # in the sweep exhausts RAM well before 20+ runs finish.
    del result["sim_data"], result["bound_data"], result["traj"]
    gc.collect()

    return result


for params in EXPERIMENTS_TO_RUN:
    result = run_current_ic_experiment(**params)
    experiment_results.append(result)
    print(f"added {result['label']} cache_hit={result.get('cache_hit')}")

running k16_eps0p0625: kmax=16, tau=1.49199e-05, nt=13710, eps=6.25e-05 kpc


TypeError: Argument 'mass' must be either a single number or a 1d array of the same length as 'pos'

In [ ]:
new_results = experiment_results[-len(EXPERIMENTS_TO_RUN):] if EXPERIMENTS_TO_RUN else []

for result in new_results:
    if "mass_loss" not in result:
        print(f"no mass-loss curve for {result['label']}")
        continue
    curve = result["mass_loss"]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(curve["elapsed_time_Myr"], curve["bound_mass_fraction"])
    ax.set_xlabel("elapsed time [Myr]")
    ax.set_ylabel(r"$M/M_0$")
    ymin, ymax = curve["bound_mass_fraction"].min(), curve["bound_mass_fraction"].max()
    pad = max((ymax - ymin) * 0.1, 1e-4)
    ax.set_ylim(ymin - pad, ymax + pad)
    ax.set_title(f"Mass-Loss History: {result['label']}")
    fig.tight_layout()
    plt.show()
